### Markers for TAL subregions

In [ ]:
from pathlib import Path
import os

import scanpy as sc
import numpy as np
import pandas as pd
import gseapy as gp

In [ ]:
adata = sc.read("data/kpmp_tal_filtered.h5ad")
adata = adata.raw.to_adata()
adata

In [ ]:
adata.var_names_make_unique()

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="subclass.l2",
    method="wilcoxon",
    pts=True,
    key_added="wilcoxon",
)

### Ranking genes

We rank genes by Wilcoxon test for effect size & p-value.


In [ ]:
marker_list = []
for grp in adata.obs["subclass.l2"].cat.categories:
    df_w = sc.get.rank_genes_groups_df(adata, group=grp, key="wilcoxon")
    df_w["group"] = grp
    marker_list.append(df_w)
df_markers = pd.concat(marker_list, ignore_index=True)
df_markers["delta_pct"] = df_markers["pct_nz_group"] - df_markers["pct_nz_reference"]

In [ ]:
print(df_markers.head())

In [ ]:
criteria = (
    (df_markers["logfoldchanges"] >= 1)
    & (df_markers["pvals_adj"] < 0.05)
    & (df_markers["delta_pct"] >= 0.25)
)
df_sig = df_markers[criteria].copy()
print(f"Found {len(df_sig)} significant markers across all groups.")
print(df_sig.head())

### Jaccard similarity check

We check the stability of markers across donors by calculating the Jaccard similarity between the full set of markers and the set of markers identified by leave-one-out cross-validation.


In [ ]:
def jaccard(set1, set2):
    set1, set2 = set(set1), set(set2)
    return len(set1 & set2) / len(set1 | set2) if set1 and set2 else np.nan


full_markers = {
    grp: set(df_sig.loc[df_sig["group"] == grp, "names"])
    for grp in adata.obs["subclass.l2"].cat.categories
}
jaccard_scores = []
for donor in adata.obs["donor_id"].unique():
    ad_train = adata[adata.obs["donor_id"] != donor].copy()
    sc.tl.rank_genes_groups(
        ad_train,
        groupby="subclass.l2",
        method="wilcoxon",
        pts=True,
        key_added="wilcox_cv",
    )

    df_list = []
    for grp in ad_train.obs["subclass.l2"].cat.categories:
        df_w = sc.get.rank_genes_groups_df(ad_train, group=grp, key="wilcox_cv")
        df_w["delta_pct"] = df_w["pct_nz_group"] - df_w["pct_nz_reference"]
        df_w["group"] = grp
        df_list.append(df_w)

    df_cv = pd.concat(df_list, ignore_index=True)
    df_cv_sig = df_cv[
        (df_cv["logfoldchanges"] >= 1)
        & (df_cv["pvals_adj"] < 0.05)
        & (df_cv["delta_pct"] >= 0.25)
    ]

    for grp in full_markers:
        j = jaccard(
            full_markers[grp], df_cv_sig.loc[df_cv_sig["group"] == grp, "names"]
        )
        jaccard_scores.append(j)
        print(
            f"Donor {donor}, group {grp}: Jaccard similarity = {j:.2f} "
            f"(n_full={len(full_markers[grp])}, n_cv={len(df_cv_sig.loc[df_cv_sig['group'] == grp, 'names'])})"
        )

mean_jaccard = np.nanmean(jaccard_scores)
print(f"Mean leave-one-out Jaccard similarity: {mean_jaccard:.2f}")

In [ ]:
top_genes = df_sig.groupby("group")["names"].apply(lambda x: x.unique().tolist())
print("Top genes per group:")
for group, genes in top_genes.items():
    print(f"{group}: {', '.join(genes[:5])}... (total {len(genes)} genes)")

In [ ]:
for grp, ensg_list in top_genes.items():
    feat_names = adata.var.loc[ensg_list, "feature_name"].values.tolist()

    sc.pl.dotplot(
        adata,
        var_names=feat_names,
        groupby="subclass.l2",
        standard_scale="var",
        gene_symbols="feature_name",
        dot_max=0.7,
        title=f"{grp} biomarkers",
        save=f"dotplot_{grp}.png",
    )

In [ ]:
top_ensgs = df_sig["names"].unique().tolist()
top_genes = [
    adata.var.loc[gid, "feature_name"] if gid in adata.var.index else gid
    for gid in top_ensgs
]

enr = gp.enrichr(
    gene_list=top_genes,
    gene_sets=[
        "GO_Biological_Process_2025",
        "GO_Cellular_Component_2025",
        "KEGG_2019_Human",
        "Reactome_Pathways_2024",
        "WikiPathways_2024_Human",
        "MSigDB_Hallmark_2020",
        "MSigDB_Oncogenic_Signatures",
        "CellMarker_2024",
        "PanglaoDB_Augmented_2021",
        "GTEx_Tissues_V8_2023",
        "DisGeNET",
        "Drug_Perturbations_from_GEO_up",
        "Drug_Perturbations_from_GEO_down",
    ],
    organism="Human",
    cutoff=0.05,
)

enrichment_results_path = Path("results/enrichment_results.csv")
os.makedirs(enrichment_results_path.parent, exist_ok=True)
enr.results.to_csv(enrichment_results_path, index=False)